In [1]:
"""
exp06 Step 02: Continuous Replay On The Validation Span (PROTOCOL.md §6-§9; DATA-DEPENDENT FOLLOW-UP OF exp03)

One continuous path over the 44 validation quarters of the exp02-exp05 schedule (2015-04-17 -> 2026-04-15); the position
carries across quarters and nothing forces a buy-back.
    1. CANDIDATE PATHS (after the fact, optimistic for the best): the 12 candidates, each replayed over the whole span; its
       return in each period minus buy-and-hold's = its period score (12 x 44 = 528 validation trials).
    2. SELECTION: mean period score over the last 4 periods; ties: longer cooling-off, then smaller buy-stop, then the
       trend exit (the choices closest to buy-and-hold).
    3. PRIOR-ONLY PATH (the honest estimate): the candidate selected at period f governs period f + 1; one continuous
       switching path from period 1 to the end, with buy-and-hold, 20 random-exit and 20 random-re-entry runs on the
       same sessions, the success criteria, the log-excess interval, the cost sensitivity, the episode scorecard and the
       roadmap arithmetic (1 - q) x G vs q x |L|.

STOP (PROTOCOL.md §9) if the prior-only path does not beat buy-and-hold, or does not beat the median of both random
families. No test window is evaluated: the bars are cut at the last validation session before any simulation.
RUN_MODE: "validation_only" only (the untouched window 2026-05-14 -> 2026-08-13 is never read for evaluation).
Prerequisite: the raw data only. Outputs: store04_experiments/exp06_capped_regret_reentry/step02_continuous_replay_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import os
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp06_capped_regret_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE CONTINUOUS REPLAY AND WALK-FORWARD FUNCTIONS
from so.core.continuous_replay import get_replay_period_pdf
from so.core.replay_walk_forward import run_candidate_replay_dict, run_prior_only_replay_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp06_capped_regret_reentry.rules import get_schedule_tuple, get_rule_candidate_list, get_rule_builder_func, get_trigger_summary_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# DEFINE THE CANDIDATE KEYS AND THE TIE-BREAKS OF THE SELECTION (PROTOCOL.md §6: LONGER COOLING-OFF, SMALLER BUY-STOP, TREND EXIT)
KEY_COL_STR_LIST = ["exit_signal", "buy_stop", "cooling"]
TIE_BREAK_LIST = [("cooling", False), ("buy_stop", True), ("exit_order", True)]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tExit signals:\t{list(exp_config.EXIT_SIGNAL_ORDER_DICT)}\tBuy-stops:\t{exp_config.BUY_STOP_LIST}\tCooling-offs:\t{exp_config.COOLING_LIST}\tMax cash sessions:\t{exp_config.MAX_CASH_SESSIONS}")

Run Mode:	validation_only	Exit signals:	['trend_ma200', 'ath_1pct']	Buy-stops:	[0.01, 0.02, 0.03]	Cooling-offs:	[0, 20]	Max cash sessions:	None


In [4]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_continuous_replay_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

Experiment:	exp06_capped_regret_reentry	Protocol:	1.0	Config Hash:	5927a03c1d
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp06_capped_regret_reentry/step02_continuous_replay_data/
ℹ️ No earlier 'validation' entry of exp06_capped_regret_reentry in the trial log (config hash 5927a03c1d).


In [5]:
# CALL FUNCTION TO READ THE MINUTE BARS AND BUILD THE SCHEDULE FROM THE FULL SESSION CALENDAR (SAME 44 QUARTERS AS exp02-exp05)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
fold_pdf, _ = get_schedule_tuple(sorted(complete_ohlcv_pdf["date"].unique()))
# DEFINE THE LAST VALIDATION SESSION AND CUT THE BARS THERE (NO LATER BAR IS GIVEN TO THE SIMULATOR)
replay_end_date = pd.to_datetime(fold_pdf["valid_end"].iloc[-1]).date()
assert replay_end_date < pd.Timestamp(exp_config.UNTOUCHED_START_DATE_STR).date()
replay_ohlcv_pdf = complete_ohlcv_pdf[complete_ohlcv_pdf["date"] <= replay_end_date].reset_index(drop=True)
del complete_ohlcv_pdf
# CALL FUNCTION TO CONVERT THE CUT DATA TO ARRAYS, BUILD THE DAILY TABLE AND THE REPLAY PERIODS
ohlcv_array_dict = get_ohlcv_array_dict(replay_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
period_pdf = get_replay_period_pdf(fold_pdf, ohlcv_array_dict["session_date_list"])
# CHECK THAT THE VALIDATION QUARTERS ARE THOSE OF exp03 (ITS SAVED SCHEDULE, IF PRESENT)
exp03_schedule_path_str = f"{paths.get_experiment_data_path_str('exp03_ath_exit', 'step02_walk_forward_data')}fold_schedule_data.csv"
if os.path.exists(exp03_schedule_path_str):
    exp03_fold_pdf = pd.read_csv(exp03_schedule_path_str)
    same_quarters_bool = exp03_fold_pdf["valid_start"].astype(str).tolist() == fold_pdf["valid_start"].astype(str).tolist() and exp03_fold_pdf["valid_end"].astype(str).tolist() == fold_pdf["valid_end"].astype(str).tolist()
    print(f"Same validation quarters as exp03:\t{same_quarters_bool}")
    assert same_quarters_bool
# SAVE THE SCHEDULE AND THE PERIODS
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
write_csv_file_to_path(period_pdf, f"{output_path_str}replay_period_data.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"Bars given to the simulator:\t{replay_ohlcv_pdf['date'].min()} -> {replay_ohlcv_pdf['date'].max()}\t({len(daily_pdf):,} sessions)")
print(f"Folds:\t{len(fold_pdf)}\tPeriods:\t{len(period_pdf)}\t{period_pdf['period_start'].iloc[0]} -> {period_pdf['period_end'].iloc[-1]}\t({int(period_pdf['session_count'].sum()):,} sessions)")
# PREVIEW DATAFRAME
period_pdf.head()

Same validation quarters as exp03:	True
Bars given to the simulator:	2005-01-03 -> 2026-04-15	(5,353 sessions)
Folds:	44	Periods:	44	2015-04-17 -> 2026-04-15	(2,765 sessions)


,period_id,period_start,period_end,session_count,valid_start,valid_end
0,0,2015-04-17,2015-07-16,63,2015-04-17,2015-07-16
1,1,2015-07-17,2015-10-14,63,2015-07-17,2015-10-16
2,2,2015-10-15,2016-01-15,64,2015-10-15,2016-01-14
3,3,2016-01-19,2016-04-15,62,2016-01-19,2016-04-15
4,4,2016-04-18,2016-07-14,62,2016-04-18,2016-07-15


In [6]:
"""
Candidate paths (AFTER THE FACT: the best of 12 chosen with hindsight is optimistic).
"""
# DEFINE THE CANDIDATES AND THE RULE BUILDER
candidate_list = get_rule_candidate_list()
rule_builder_func = get_rule_builder_func(daily_pdf)
# CALL FUNCTION TO REPLAY EVERY CANDIDATE AND SCORE IT PER PERIOD
start_time = time.time()
candidate_replay_dict = run_candidate_replay_dict(ohlcv_array_dict, period_pdf, candidate_list, rule_builder_func, exp_config.UNTOUCHED_START_DATE_STR,
                                                  exp_config.MAX_CASH_SESSIONS, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
candidate_period_pdf, candidate_summary_pdf = candidate_replay_dict["candidate_period_pdf"], candidate_replay_dict["candidate_summary_pdf"]
print(f"Candidates replayed in {time.time() - start_time:.0f} s")
# LOG EVERY (CANDIDATE, PERIOD) AS A VALIDATION TRIAL (ONE WRITE)
validation_trial_pdf = log_trial_pdf(exp_config, "validation", candidate_period_pdf, KEY_COL_STR_LIST, [col for col in candidate_period_pdf.columns if col.startswith("valid_")],
                                     False, fold_pdf, note_str_in="exp06 step02 candidate path, one period of the continuous replay")
print(f"Validation trials logged:\t{len(validation_trial_pdf)}")
# SAVE THE CANDIDATES
write_csv_file_to_path(candidate_period_pdf, f"{output_path_str}candidate_period_data.csv", "W", alert_in=False)
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# DISPLAY THE EXCESS TOTAL RETURN GRID (EXIT SIGNAL AND BUY-STOP x COOLING-OFF)
display(candidate_summary_pdf.pivot_table(index=["exit_signal", "buy_stop"], columns="cooling", values="excess_total_return").round(4))
# PREVIEW DATAFRAME
candidate_summary_pdf[["candidate_idx", "exit_signal", "buy_stop", "cooling", "strategy_total_return", "buy_hold_total_return", "excess_total_return", "strategy_annualized_return",
                       "buy_hold_annualized_return", "in_market_pct", "exit_count", "mean_cash_sessions", "bought_back_lower_pct", "period_windows_beating_buy_hold",
                       "log_excess_annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high", "strategy_max_drawdown", "buy_hold_max_drawdown"]].round(4)

	candidate 0 {'exit_signal': 'trend_ma200', 'exit_order': 0, 'buy_stop': 0.01, 'cooling': 0}: total 113.07% vs buy & hold 235.39% | in market 80.5% | exits 40


	candidate 1 {'exit_signal': 'trend_ma200', 'exit_order': 0, 'buy_stop': 0.01, 'cooling': 20}: total 144.80% vs buy & hold 235.39% | in market 87.2% | exits 27


	candidate 2 {'exit_signal': 'trend_ma200', 'exit_order': 0, 'buy_stop': 0.02, 'cooling': 0}: total 108.03% vs buy & hold 235.39% | in market 80.2% | exits 35


	candidate 3 {'exit_signal': 'trend_ma200', 'exit_order': 0, 'buy_stop': 0.02, 'cooling': 20}: total 101.75% vs buy & hold 235.39% | in market 85.4% | exits 33


	candidate 4 {'exit_signal': 'trend_ma200', 'exit_order': 0, 'buy_stop': 0.03, 'cooling': 0}: total 108.03% vs buy & hold 235.39% | in market 80.2% | exits 35


...

	candidate 9 {'exit_signal': 'ath_1pct', 'exit_order': 1, 'buy_stop': 0.02, 'cooling': 20}: total 156.26% vs buy & hold 235.39% | in market 65.4% | exits 38


	candidate 10 {'exit_signal': 'ath_1pct', 'exit_order': 1, 'buy_stop': 0.03, 'cooling': 0}: total 69.98% vs buy & hold 235.39% | in market 39.4% | exits 45


	candidate 11 {'exit_signal': 'ath_1pct', 'exit_order': 1, 'buy_stop': 0.03, 'cooling': 20}: total 113.42% vs buy & hold 235.39% | in market 61.7% | exits 33
Candidates replayed in 11 s
Validation trials logged:	528


cooling                   0       20
exit_signal buy_stop                
ath_1pct    0.01     -1.6673 -0.3527
            0.02     -1.5991 -0.7913
            0.03     -1.6541 -1.2197
trend_ma200 0.01     -1.2232 -0.9059
            0.02     -1.2736 -1.3364
            0.03     -1.2736 -1.3101

,candidate_idx,exit_signal,buy_stop,cooling,strategy_total_return,buy_hold_total_return,excess_total_return,strategy_annualized_return,buy_hold_annualized_return,in_market_pct,exit_count,mean_cash_sessions,bought_back_lower_pct,period_windows_beating_buy_hold,log_excess_annualized_log_excess,log_excess_ci_low,log_excess_ci_high,strategy_max_drawdown,buy_hold_max_drawdown
0,0,trend_ma200,0.01,0,1.1307,2.3539,-1.2232,0.0714,0.1166,0.8050,40,13.4750,0.0750,18,-0.0409,-0.0794,-0.0042,-0.1894,-0.342
1,1,trend_ma200,0.01,20,1.4480,2.3539,-0.9059,0.0850,0.1166,0.8723,27,13.0741,0.0741,12,-0.0284,-0.0612,0.0054,-0.3640,-0.342
2,2,trend_ma200,0.02,0,1.0803,2.3539,-1.2736,0.0690,0.1166,0.8021,35,15.6286,0.0857,12,-0.0431,-0.0819,-0.0047,-0.1996,-0.342
3,3,trend_ma200,0.02,20,1.0175,2.3539,-1.3364,0.0661,0.1166,0.8538,33,12.2424,0.0909,10,-0.0459,-0.0850,-0.0110,-0.3666,-0.342
4,4,trend_ma200,0.03,0,1.0803,2.3539,-1.2736,0.0690,0.1166,0.8021,35,15.6286,0.0857,12,-0.0431,-0.0819,-0.0047,-0.1996,-0.342
5,5,trend_ma200,0.03,20,1.0438,2.3539,-1.3101,0.0673,0.1166,0.8205,35,14.1714,0.0857,11,-0.0447,-0.0825,-0.0102,-0.3639,-0.342
6,6,ath_1pct,0.01,0,0.6867,2.3539,-1.6673,0.0488,0.1166,0.4323,104,15.2233,0.0962,20,-0.0620,-0.1185,-0.0093,-0.2993,-0.342
7,7,ath_1pct,0.01,20,2.0012,2.3539,-0.3527,0.1054,0.1166,0.7059,45,18.4545,0.1778,19,-0.0100,-0.0413,0.0218,-0.2885,-0.342
8,8,ath_1pct,0.02,0,0.7548,2.3539,-1.5991,0.0526,0.1166,0.4153,59,27.8448,0.1695,20,-0.0584,-0.1077,-0.0090,-0.2889,-0.342
9,9,ath_1pct,0.02,20,1.5626,2.3539,-0.7913,0.0896,0.1166,0.6538,38,25.8378,0.2368,20,-0.0243,-0.0603,0.0122,-0.2887,-0.342


In [7]:
"""
Prior-only path (the honest validation estimate) with its baselines on the same sessions.
"""
# CALL FUNCTION TO SELECT, REPLAY THE SWITCHING PATH AND RUN THE BASELINES
start_time = time.time()
prior_dict = run_prior_only_replay_dict(ohlcv_array_dict, period_pdf, candidate_period_pdf, candidate_list, rule_builder_func, KEY_COL_STR_LIST, TIE_BREAK_LIST,
                                        exp_config.SELECTION_POOLED_QUARTER_COUNT, exp_config.UNTOUCHED_START_DATE_STR, exp_config.MAX_CASH_SESSIONS,
                                        exp_config.RANDOM_RUN_COUNT, exp_config.RANDOM_EXIT_SEED_OFFSET, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
summary_dict, baseline_summary_dict, log_excess_1m_dict = prior_dict["summary_dict"], prior_dict["baseline_summary_dict"], prior_dict["log_excess_1m_dict"]
print(f"Prior-only path and baselines in {time.time() - start_time:.0f} s")
# SAVE THE PATH
write_csv_file_to_path(prior_dict["selection_pdf"], f"{output_path_str}selection_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["baseline_pdf"], f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["scorecard_pdf"], f"{output_path_str}prior_only_scorecard_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["daily_equity_pdf"], f"{output_path_str}prior_only_daily_equity_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["transaction_pdf"], f"{output_path_str}prior_only_transaction_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["cost_sensitivity_pdf"], f"{output_path_str}prior_only_cost_sensitivity_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}]), f"{output_path_str}prior_only_summary_data.csv", "W", alert_in=False)
# DISPLAY THE PATH AGAINST BUY AND HOLD
print(f"\nPrior-only path:\t{summary_dict['first_date']} -> {summary_dict['last_date']}\t({summary_dict['session_count']:,} sessions, {summary_dict['year_count']:.2f} years)")
print(f"Total return:\tpath {summary_dict['strategy_total_return']:.2%} | buy & hold {summary_dict['buy_hold_total_return']:.2%}\t(annualized {summary_dict['strategy_annualized_return']:.2%} vs {summary_dict['buy_hold_annualized_return']:.2%})")
print(f"Time in the market:\t{summary_dict['in_market_pct']:.1%}\tExits:\t{prior_dict['simulation_dict']['metric_dict']['signal_exit_count']}\tMean cash decisions per episode:\t{prior_dict['simulation_dict']['metric_dict']['mean_cash_sessions']:.1f}")
print(f"Periods won:\t{summary_dict['period_windows_beating_buy_hold']} of {summary_dict['period_window_count']}\t(sign test p = {summary_dict['period_sign_test_p_value']:.3f})"
      f"\tmean period excess {summary_dict['period_mean_excess_return']:.2%} (SD {summary_dict['period_std_excess_return']:.2%}, t = {summary_dict['period_t_stat']:.2f})"
      f"\tMDE {summary_dict['period_mde_per_window']:.2%} per period ({summary_dict['period_mde_per_year']:.2%} per year)")
print(f"Annualized log excess:\t{summary_dict['log_excess_annualized_log_excess']:.4f}\t95% interval (6-month blocks) [{summary_dict['log_excess_ci_low']:.4f}, {summary_dict['log_excess_ci_high']:.4f}]"
      f"\t(1-month blocks [{log_excess_1m_dict['ci_low']:.4f}, {log_excess_1m_dict['ci_high']:.4f}])\tsupported: {summary_dict['log_excess_supported']}")
print(f"Sharpe:\tpath {summary_dict['strategy_sharpe_ratio']:.2f} | buy & hold {summary_dict['buy_hold_sharpe_ratio']:.2f}\tMax drawdown:\tpath {summary_dict['strategy_max_drawdown']:.2%} | buy & hold {summary_dict['buy_hold_max_drawdown']:.2%}")
print(f"PRIMARY (beats buy & hold):\t{summary_dict['primary_success']}\tComparable:\t{summary_dict['comparable_return']}\tSecondary Sharpe / drawdown:\t{summary_dict['secondary_sharpe']} / {summary_dict['secondary_drawdown']}")
print(f"Random exit (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_exit_median']:.2%}\tpath beats {baseline_summary_dict['random_exit_path_beats_share']:.0%} of runs")
print(f"Random re-entry (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_reentry_median']:.2%}\tpath beats {baseline_summary_dict['random_reentry_path_beats_share']:.0%} of runs")
print(f"Matched probabilities:\t{prior_dict['probability_dict']}")
# APPLY THE STOPPING RULE (PROTOCOL.md §9)
information_bool = baseline_summary_dict["random_exit_path_beats_share"] > 0.5 and baseline_summary_dict["random_reentry_path_beats_share"] > 0.5 and summary_dict["all_baselines_beaten"]
verdict_str = "CANDIDATE RESULT PENDING REVIEW" if summary_dict["primary_success"] and information_bool else "STOP"
print(f"\nInformation test passed:\t{information_bool}\tVERDICT:\t{verdict_str}")
# DISPLAY THE RE-ENTRY TRIGGERS AND THE ROADMAP ARITHMETIC OF THE PATH ((1 - q) x G vs q x |L|, LOG SHARE GAINS)
trigger_dict = get_trigger_summary_dict(prior_dict["simulation_dict"])
print(f"Episodes:\t{trigger_dict['episode_count']}\tbuy-stop:\t{trigger_dict['buy_stop_count']}\trecovery:\t{trigger_dict['recovery_count']}\topen at the end:\t{trigger_dict['open_count']}")
print(f"Wrong-exit share q:\t{trigger_dict['wrong_exit_share']:.3f}\tmean gain when right G:\t{trigger_dict['mean_gain_right']:.4f}\tmean loss when wrong L:\t{trigger_dict['mean_loss_wrong']:.4f}"
      f"\t(1 - q) x G = {trigger_dict['right_side']:.4f} vs q x |L| = {trigger_dict['wrong_side']:.4f}")
# PREVIEW THE COST SENSITIVITY
prior_dict["cost_sensitivity_pdf"].round(4)

Prior-only path and baselines in 35 s

Prior-only path:	2015-07-17 -> 2026-04-15	(2,702 sessions, 10.72 years)
Total return:	path 147.38% | buy & hold 229.74%	(annualized 8.81% vs 11.77%)
Time in the market:	69.8%	Exits:	45	Mean cash decisions per episode:	18.1
Periods won:	13 of 43	(sign test p = 0.997)	mean period excess -0.79% (SD 4.25%, t = -1.21)	MDE 1.81% per period (7.26% per year)
Annualized log excess:	-0.0265	95% interval (6-month blocks) [-0.0674, 0.0153]	(1-month blocks [-0.0799, 0.0307])	supported: False
Sharpe:	path 0.66 | buy & hold 0.71	Max drawdown:	path -28.84% | buy & hold -34.21%
PRIMARY (beats buy & hold):	False	Comparable:	False	Secondary Sharpe / drawdown:	False / False
Random exit (median of 20):	124.82%	path beats 95% of runs
Random re-entry (median of 20):	103.08%	path beats 75% of runs
Matched probabilities:	{'exit_probability': 0.023851190705129467, 'reentry_probability': 0.05521472392638037}

Information test passed:	True	VERDICT:	STOP
Episodes:	45	buy-stop

,slippage_per_share,total_return,buy_hold_return,excess_return
0,0.00,1.4805,2.2975,-0.8170
1,0.01,1.4738,2.2974,-0.8236
2,0.02,1.4675,2.2973,-0.8298


In [8]:
"""
Episode scorecard of the prior-only path (every exit). S / R > 1 = bought back lower than sold.
"""
# DISPLAY THE SCORECARD
scorecard_pdf = prior_dict["scorecard_pdf"]
if not scorecard_pdf.empty:
    print(f"Episodes:\t{len(scorecard_pdf)}\tbought back lower:\t{int((scorecard_pdf['s_over_r'] > 1).sum())}\tproduct of S/R:\t{np.prod(scorecard_pdf['s_over_r']):.4f}"
          f"\tmean sessions out:\t{scorecard_pdf['sessions_out'].mean():.1f}\ttotal cost:\t{scorecard_pdf['cost_paid_pct'].sum():.4%} of the sale values")
scorecard_pdf[["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "buy_hold_return_out", "max_decline_out_pct", "cost_paid_pct", "open_at_end"]].round(4)

Episodes:	45	bought back lower:	6	product of S/R:	0.7514	mean sessions out:	18.1	total cost:	0.3710% of the sale values


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
0,2015-07-17,212.40,2015-10-23,207.59,1.0232,69,-0.0227,-0.1413,0.0001,False
1,2016-06-07,211.64,2016-06-28,203.12,1.0419,15,-0.0403,-0.0614,0.0001,False
2,2016-07-08,212.71,2016-07-15,215.89,0.9853,5,0.0149,-0.0005,0.0001,False
3,2016-08-15,219.11,2016-11-25,221.48,0.9893,72,0.0107,-0.0490,0.0001,False
4,2016-12-27,226.16,2017-01-25,229.57,0.9851,19,0.0150,-0.0152,0.0001,False
5,2017-02-24,236.60,2017-03-01,239.86,0.9864,3,0.0137,-0.0025,0.0001,False
6,2017-04-25,238.68,2017-06-02,244.00,0.9782,27,0.0222,-0.0137,0.0001,False
7,2017-07-05,242.89,2017-08-07,247.86,0.9799,23,0.0204,-0.0105,0.0001,False
8,2018-04-02,257.51,2018-04-03,260.63,0.9880,1,0.0120,-0.0026,0.0001,False
9,2018-10-11,272.50,2018-10-12,276.64,0.9850,1,0.0151,-0.0011,0.0001,False


In [9]:
# DISPLAY THE SELECTED CANDIDATE PER PERIOD (THE CHOICE OF PERIOD f GOVERNS PERIOD f + 1)
selection_pdf = prior_dict["selection_pdf"]
print("Selections (candidate: periods):", selection_pdf.groupby(["exit_signal", "buy_stop", "cooling"]).size().to_dict())
selection_pdf[["fold_id", "exit_signal", "buy_stop", "cooling", "pooled_score", "pooled_quarter_count"]].tail(12).round(4)

Selections (candidate: periods): {('ath_1pct', 0.01, 0): 1, ('ath_1pct', 0.01, 20): 15, ('ath_1pct', 0.02, 20): 6, ('ath_1pct', 0.03, 0): 4, ('trend_ma200', 0.01, 0): 6, ('trend_ma200', 0.01, 20): 7, ('trend_ma200', 0.02, 0): 1, ('trend_ma200', 0.02, 20): 4}


,fold_id,exit_signal,buy_stop,cooling,pooled_score,pooled_quarter_count
32,32,ath_1pct,0.03,0,0.0000,4
33,33,ath_1pct,0.03,0,0.0000,4
34,34,ath_1pct,0.03,0,-0.0006,4
35,35,ath_1pct,0.01,20,0.0017,4
36,36,trend_ma200,0.01,0,-0.0050,4
37,37,trend_ma200,0.01,0,-0.0050,4
38,38,trend_ma200,0.01,0,-0.0000,4
39,39,trend_ma200,0.02,0,0.0124,4
40,40,ath_1pct,0.01,20,0.0030,4
41,41,ath_1pct,0.01,20,0.0007,4


In [10]:
# CREATE A FIGURE OF THE PRIOR-ONLY PATH vs BUY AND HOLD (GROWTH OF 1)
path_equity_pdf, buy_hold_equity_pdf = prior_dict["simulation_dict"]["daily_equity_pdf"], prior_dict["buy_hold_dict"]["daily_equity_pdf"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=path_equity_pdf["date"], y=path_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="prior-only path", line={"width": 1.5}))
fig.add_trace(go.Scatter(x=buy_hold_equity_pdf["date"], y=buy_hold_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="buy & hold", line={"width": 1.5}))
fig.update_layout(template="plotly_dark", width=1100, height=450, title="exp06 prior-only path vs buy-and-hold (continuous replay, growth of 1)")
fig.show()

In [11]:
# LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATE PERIODS ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": f"pooled {exp_config.SELECTION_POOLED_QUARTER_COUNT} periods", "path": "prior-only continuous replay"},
                                    {**{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}, **baseline_summary_dict,
                                     "log_excess_1m_ci_low": log_excess_1m_dict["ci_low"], "log_excess_1m_ci_high": log_excess_1m_dict["ci_high"],
                                     "exit_count": prior_dict["simulation_dict"]["metric_dict"]["signal_exit_count"], **{f"trigger_{key}": value for key, value in trigger_dict.items()}, "verdict": verdict_str,
                                     "best_after_the_fact_total_return": float(candidate_summary_pdf["strategy_total_return"].max())},
                                    False, note_str_in="exp06 step02 validation summary (prior-only continuous path, baselines, MDE, log-excess interval)")
print(f"Summary logged:\t{summary_trial_dict['trial_id']}")

Summary logged:	20261006144822024098_5927a03c1d


In [12]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
1,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
2,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
3,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
4,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
5,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
6,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
7,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
8,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
9,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
